# AE 696 Lab 3
## Deep Q Network on CartPole

**Name:** Replace with your name  
**Red ID:** Replace with your Red ID  
**Due date:** See Canvas

### Purpose

In this lab, you will implement the essential components of a Deep Q Network, DQN, and train an agent on `CartPole-v1`. The lab emphasizes the connection between the DQN target from Lecture 14 and its PyTorch implementation.

### Submission

Submit one completed Jupyter notebook, named `AE696_Lab3_Lastname_Firstname.ipynb`. Do not submit a ZIP file.

Before submitting:

1. Restart the kernel.
2. Run all cells from top to bottom.
3. Confirm that the final grading summary appears.
4. Save the notebook with all outputs visible.

### Grading

| Part | Topic | Points |
|:---:|---|---:|
| A | Environment dimensions | 5 |
| B | Q network | 15 |
| C | Replay buffer | 15 |
| D | Epsilon greedy action selection | 10 |
| E | Terminal masked DQN target | 20 |
| F | One optimization step | 20 |
| G | Training and evaluation outputs | 10 |
| H | Objective concept check | 5 |
|  | **Total** | **100** |

Cells labeled **CHECK** compute the public score. Do not edit those cells. Training performance is reported for diagnosis, but it is not used to assign points because deep RL results vary across machines and random seeds.

## Setup

This notebook requires Python 3.10 or newer and the packages below. If an import fails, uncomment and run the installation line once, then restart the kernel.

```python
%pip install numpy matplotlib gymnasium torch
```

In [1]:
# Core imports
import math
import random
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Gymnasium version:", gym.__version__)
print("PyTorch version:", torch.__version__)
print("Device:", device)


Gymnasium version: 1.3.0
PyTorch version: 2.14.1+cu130
Device: cpu


In [2]:
# Public check infrastructure, do not edit
LAB3_CHECKS = {}

def record_check(name, passed, points, note=""):
    passed = bool(passed)
    LAB3_CHECKS[name] = {"passed": passed, "points": int(points), "note": note}
    status = "PASS" if passed else "FAIL"
    print(f"{status}: {name}, {points} point(s)")
    if note and not passed:
        print("  ", note)


## Part A Environment dimensions, 5 points

CartPole observations contain four continuous values. The environment has two discrete actions.

Complete the two assignments below using the Gymnasium space objects. Do not type the dimensions as fixed numbers.

In [3]:
env = gym.make("CartPole-v1")
observation, info = env.reset(seed=SEED)
env.action_space.seed(SEED)

# TODO A1 and A2, obtain both dimensions from the environment
STATE_DIM = env.observation_space.shape[0]
ACTION_DIM = env.action_space.n



print("Initial observation:", observation)
print("State dimension:", STATE_DIM)
print("Action dimension:", ACTION_DIM)
env.close()


Initial observation: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
State dimension: 4
Action dimension: 2


In [4]:
# CHECK A, 5 points, do not edit
record_check("A1 state dimension", STATE_DIM == 4, 2,
             "STATE_DIM must come from env.observation_space.")
record_check("A2 action dimension", ACTION_DIM == 2, 3,
             "ACTION_DIM must come from env.action_space.")


PASS: A1 state dimension, 2 point(s)
PASS: A2 action dimension, 3 point(s)


## Part B Q network, 15 points

Define a neural network that maps a batch of states to one Q value per action.

Required architecture:

1. Linear layer from `state_dim` to `hidden_dim`
2. ReLU
3. Linear layer from `hidden_dim` to `hidden_dim`
4. ReLU
5. Linear layer from `hidden_dim` to `action_dim`

Do not place an activation after the final layer because Q values are not probabilities.

In [5]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
# CHECK B, 15 points, do not edit
try:
    test_net = QNetwork(STATE_DIM, ACTION_DIM, hidden_dim=64).to(device)
    single_output = test_net(torch.zeros(1, STATE_DIM, device=device))
    batch_output = test_net(torch.zeros(7, STATE_DIM, device=device))
    parameter_count = sum(p.numel() for p in test_net.parameters() if p.requires_grad)

    record_check("B1 single state output shape", single_output.shape == (1, ACTION_DIM), 6,
                 "Expected shape (1, 2).")
    record_check("B2 batch output shape", batch_output.shape == (7, ACTION_DIM), 6,
                 "The first dimension must remain the batch dimension.")
    record_check("B3 trainable parameters", parameter_count > 0, 3,
                 "The network must contain trainable layers.")
except Exception as exc:
    record_check("B1 single state output shape", False, 6, str(exc))
    record_check("B2 batch output shape", False, 6, str(exc))
    record_check("B3 trainable parameters", False, 3, str(exc))


PASS: B1 single state output shape, 6 point(s)
PASS: B2 batch output shape, 6 point(s)
PASS: B3 trainable parameters, 3 point(s)


## Part C Replay buffer, 15 points

The replay buffer stores transitions and returns random minibatches. Each transition contains

\[
(s, a, r, s', d),
\]

where `d` records true environment termination. A time limit truncation ends the rollout, but it does not set `d = 1` for the DQN target.

In [7]:
Transition = namedtuple(
    "Transition", ["state", "action", "reward", "next_state", "terminated"]
)

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, terminated):
        # TODO C1, store one transition

        self.buffer.append(Transition(
            state, 
            action, 
            reward, 
            next_state, 
            terminated
            ))

    def sample(self, batch_size):
        # TODO C2, sample without replacement and return five NumPy arrays
        sampled_list = random.sample(self.buffer, k=batch_size)

        States, Actions, Rewards, Next_states, Terminates = zip(*sampled_list)

        return States, Actions, Rewards, Next_states, Terminates


    def __len__(self):
        return len(self.buffer)


In [8]:
# CHECK C, 15 points, do not edit
try:
    test_buffer = ReplayBuffer(capacity=5)
    for index in range(7):
        state = np.full(STATE_DIM, index, dtype=np.float32)
        next_state = state + 1.0
        test_buffer.push(state, index % ACTION_DIM, 1.0, next_state, index == 6)

    record_check("C1 capacity and length", len(test_buffer) == 5, 5,
                 "A deque with maxlen should retain the five newest transitions.")

    sample = test_buffer.sample(4)
    expected_shapes = [(4, STATE_DIM), (4,), (4,), (4, STATE_DIM), (4,)]
    actual_shapes = [np.asarray(item).shape for item in sample]
    record_check("C2 minibatch shapes", actual_shapes == expected_shapes, 7,
                 f"Expected {expected_shapes}, received {actual_shapes}.")

    stored_states = [int(item.state[0]) for item in test_buffer.buffer]
    record_check("C3 oldest transitions removed", stored_states == [2, 3, 4, 5, 6], 3,
                 "The replay buffer must discard the oldest item when full.")
    
except Exception as exc:
    record_check("C1 capacity and length", False, 5, str(exc))
    record_check("C2 minibatch shapes", False, 7, str(exc))
    record_check("C3 oldest transitions removed", False, 3, str(exc))


PASS: C1 capacity and length, 5 point(s)
PASS: C2 minibatch shapes, 7 point(s)
PASS: C3 oldest transitions removed, 3 point(s)


## Part D Epsilon greedy action selection, 10 points

With probability \(\epsilon\), choose a uniformly random action. Otherwise, select

\[
a = \arg\max_a Q_\theta(s,a).
\]

Use `torch.no_grad()` during action selection.

In [9]:
def select_action(state, q_net, epsilon, action_dim, device):
    # TODO D1, implement epsilon greedy action selection
    with torch.no_grad():
        if np.random.rand() < epsilon:
            # Explore, pick a random action
            return random.randrange(action_dim)
        else:
            # Exploit, pick greedy Q
            state_tensor = torch.as_tensor(
                state, 
                dtype=torch.float32, 
                device=device
                )
            state_batch = state_tensor.unsqueeze(0)
            q_values = q_net(state_batch)
            greedy = torch.argmax(q_values, dim=1) 
            return greedy.item()



In [ ]:
# CHECK D, 10 points, do not edit
class FixedQNetwork(nn.Module):
    def forward(self, x):
        values = torch.tensor([[-1.0, 3.0]], dtype=torch.float32, device=x.device)
        return values.repeat(x.shape[0], 1)

try:
    fixed_q = FixedQNetwork().to(device)
    greedy_action = select_action(
        np.zeros(STATE_DIM, dtype=np.float32), fixed_q, 0.0, ACTION_DIM, device
    )
    record_check("D1 greedy action", greedy_action == 1, 5,
                 "With epsilon equal to zero, select the largest Q value.")

    set_seed(101)
    random_actions = [
        select_action(np.zeros(STATE_DIM, dtype=np.float32), fixed_q, 1.0, ACTION_DIM, device)
        for _ in range(40)
    ]
    valid_exploration = set(random_actions).issubset(set(range(ACTION_DIM))) and len(set(random_actions)) == ACTION_DIM
    record_check("D2 random exploration", valid_exploration, 5,
                 "With epsilon equal to one, actions must be valid and both actions should appear.")
    set_seed()
except Exception as exc:
    record_check("D1 greedy action", False, 5, str(exc))
    record_check("D2 random exploration", False, 5, str(exc))


PASS: D1 greedy action, 5 point(s)
PASS: D2 random exploration, 5 point(s)


## Part E Terminal masked DQN target, 20 points

Implement the target from Lecture 14,


\[
y_i = r_i + \gamma(1-d_i)\max_{a'}Q_{\theta^-}(s'_i,a').
\]

Here, \(d_i=1\) only for true environment termination. If an episode ends only because of the time limit, keep \(d_i=0\), retain the bootstrap term, and still stop the rollout.

In [ ]:
def compute_dqn_targets(rewards, next_states, terminated, target_net, gamma):
    # TODO E1, compute a detached column tensor of DQN targets
    with torch.no_grad():
        q_values = target_net(next_states)
        # greedy = torch.argmax(q_values, dim=1) 
        greedy = q_values.max(dim=1, keepdim=True).values 
        # ^^^ keeps results shaped like rewards (column tensor)
        y_i = rewards + gamma * (1 - terminated) * greedy

        return y_i



In [40]:
# CHECK E, 20 points, do not edit
class ConstantTargetNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.values = nn.Parameter(torch.tensor([2.4, 1.2]), requires_grad=False)

    def forward(self, x):
        return self.values.unsqueeze(0).repeat(x.shape[0], 1)

try:
    fixed_target = ConstantTargetNetwork().to(device)
    rewards_test = torch.tensor([[1.0], [1.0], [0.5]], device=device)
    next_states_test = torch.zeros(3, STATE_DIM, device=device)
    # Rows represent continuing, terminated, and time limit truncated transitions.
    terminated_test = torch.tensor([[0.0], [1.0], [0.0]], device=device)

    targets_test = compute_dqn_targets(
        rewards_test, next_states_test, terminated_test, fixed_target, 0.99
    )
    expected_targets = torch.tensor([[3.376], [1.000], [2.876]], device=device)

    record_check("E1 numerical targets", torch.allclose(targets_test, expected_targets, atol=1e-5), 15,
                 f"Expected {expected_targets.flatten().tolist()}.")
    record_check("E2 detached targets", not targets_test.requires_grad, 5,
                 "The target network computation must use torch.no_grad().")
except Exception as exc:
    record_check("E1 numerical targets", False, 15, str(exc))
    record_check("E2 detached targets", False, 5, str(exc))


PASS: E1 numerical targets, 15 point(s)
PASS: E2 detached targets, 5 point(s)


## Part F One optimization step, 20 points

Complete one DQN update:

1. Sample a minibatch.
2. Convert all arrays to tensors on `device`.
3. Use `gather` to select \(Q_\theta(s_i,a_i)\).
4. Call `compute_dqn_targets`.
5. Compute Huber loss with `F.smooth_l1_loss`.
6. Update only the online network.
7. Clip the online gradient norm to 10.

Return the scalar loss as a Python `float`.

In [13]:
def optimize_model(
    q_net, target_net, replay_buffer, optimizer, batch_size, gamma, device
):
    if len(replay_buffer) < batch_size:
        return None

    # TODO F1, sample the buffer and build the tensors

    # TODO F2, compute current Q values and detached DQN targets

    # TODO F3, compute Huber loss and update only the online network

    raise NotImplementedError("Complete optimize_model")


In [14]:
# CHECK F, 20 points, do not edit
try:
    set_seed(202)
    update_buffer = ReplayBuffer(capacity=200)
    rng = np.random.default_rng(202)
    for index in range(96):
        state = rng.normal(size=STATE_DIM).astype(np.float32)
        next_state = (state + rng.normal(scale=0.1, size=STATE_DIM)).astype(np.float32)
        update_buffer.push(state, index % ACTION_DIM, 1.0, next_state, index % 17 == 0)

    online_test = QNetwork(STATE_DIM, ACTION_DIM, hidden_dim=32).to(device)
    target_test = QNetwork(STATE_DIM, ACTION_DIM, hidden_dim=32).to(device)
    target_test.load_state_dict(online_test.state_dict())
    test_optimizer = optim.Adam(online_test.parameters(), lr=1e-3)

    online_before = [parameter.detach().clone() for parameter in online_test.parameters()]
    target_before = [parameter.detach().clone() for parameter in target_test.parameters()]

    test_loss = optimize_model(
        online_test, target_test, update_buffer, test_optimizer, 32, 0.99, device
    )

    online_changed = any(
        not torch.equal(before, after.detach())
        for before, after in zip(online_before, online_test.parameters())
    )
    target_unchanged = all(
        torch.equal(before, after.detach())
        for before, after in zip(target_before, target_test.parameters())
    )

    record_check("F1 finite scalar loss", isinstance(test_loss, float) and np.isfinite(test_loss), 5,
                 "Return a finite Python float.")
    record_check("F2 online parameters updated", online_changed, 10,
                 "Backpropagation and optimizer.step() must update q_net.")
    record_check("F3 target parameters unchanged", target_unchanged, 5,
                 "Do not optimize target_net.")
    set_seed()
except Exception as exc:
    record_check("F1 finite scalar loss", False, 5, str(exc))
    record_check("F2 online parameters updated", False, 10, str(exc))
    record_check("F3 target parameters unchanged", False, 5, str(exc))


FAIL: F1 finite scalar loss, 5 point(s)
   Complete optimize_model
FAIL: F2 online parameters updated, 10 point(s)
   Complete optimize_model
FAIL: F3 target parameters unchanged, 5 point(s)
   Complete optimize_model


## Part G Train and evaluate, 10 points

The training loop is provided. Read it carefully before running it.

Notice the two different Boolean variables:

* `episode_end = terminated or truncated` stops the rollout.
* Only `terminated` is stored for the target mask.

Expected CPU runtime is usually a few minutes. The exact learning curve will vary.

In [15]:
def train_dqn(seed=SEED, num_episodes=200):
    set_seed(seed)
    train_env = gym.make("CartPole-v1")
    train_env.action_space.seed(seed)

    online_net = QNetwork(STATE_DIM, ACTION_DIM, hidden_dim=64).to(device)
    target_net = QNetwork(STATE_DIM, ACTION_DIM, hidden_dim=64).to(device)
    target_net.load_state_dict(online_net.state_dict())
    target_net.eval()

    replay = ReplayBuffer(capacity=20_000)
    optimizer = optim.Adam(online_net.parameters(), lr=1e-3)

    gamma = 0.99
    batch_size = 64
    warmup_steps = 500
    target_update_steps = 250
    epsilon_start = 1.0
    epsilon_end = 0.05
    epsilon_decay_steps = 8_000

    episode_returns = []
    losses = []
    total_steps = 0

    for episode in range(num_episodes):
        state, info = train_env.reset(seed=seed + episode)
        episode_return = 0.0

        for step in range(500):
            epsilon = epsilon_end + (epsilon_start - epsilon_end) * math.exp(
                -total_steps / epsilon_decay_steps
            )
            action = select_action(state, online_net, epsilon, ACTION_DIM, device)
            next_state, reward, terminated, truncated, info = train_env.step(action)
            episode_end = terminated or truncated

            # Mask bootstrapping only for true environment termination.
            replay.push(state, action, reward, next_state, terminated)

            state = next_state
            episode_return += reward
            total_steps += 1

            if total_steps >= warmup_steps:
                loss = optimize_model(
                    online_net,
                    target_net,
                    replay,
                    optimizer,
                    batch_size,
                    gamma,
                    device,
                )
                if loss is not None:
                    losses.append(loss)

            if total_steps % target_update_steps == 0:
                target_net.load_state_dict(online_net.state_dict())

            if episode_end:
                break

        episode_returns.append(episode_return)

        if (episode + 1) % 25 == 0:
            recent_mean = np.mean(episode_returns[-25:])
            print(
                f"Episode {episode + 1:3d}, "
                f"mean return over last 25 = {recent_mean:6.1f}, "
                f"epsilon = {epsilon:.3f}"
            )

    train_env.close()
    return online_net, episode_returns, losses


TRAINED_Q_NET, EPISODE_RETURNS, LOSS_HISTORY = train_dqn()


NotImplementedError: Complete optimize_model

In [ ]:
def moving_average(values, window=20):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < window:
        return np.array([]), np.array([])
    average = np.convolve(values, np.ones(window) / window, mode="valid")
    episodes = np.arange(window - 1, len(values))
    return episodes, average


ma_episodes, ma_returns = moving_average(EPISODE_RETURNS, window=20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(EPISODE_RETURNS, color="0.75", linewidth=1, label="Episode return")
axes[0].plot(ma_episodes, ma_returns, color="#a6192e", linewidth=2, label="20 episode mean")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Return")
axes[0].set_title("CartPole training return")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(LOSS_HISTORY, color="#007a78", linewidth=1)
axes[1].set_xlabel("Optimization step")
axes[1].set_ylabel("Huber loss")
axes[1].set_title("DQN optimization loss")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
def evaluate_policy(q_net, episodes=10, seed=900):
    evaluation_env = gym.make("CartPole-v1")
    returns = []

    for episode in range(episodes):
        state, info = evaluation_env.reset(seed=seed + episode)
        total_reward = 0.0

        while True:
            action = select_action(state, q_net, 0.0, ACTION_DIM, device)
            state, reward, terminated, truncated, info = evaluation_env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        returns.append(total_reward)

    evaluation_env.close()
    return returns


EVALUATION_RETURNS = evaluate_policy(TRAINED_Q_NET, episodes=10)
print("Evaluation returns:", EVALUATION_RETURNS)
print("Mean evaluation return:", float(np.mean(EVALUATION_RETURNS)))


In [ ]:
# CHECK G, 10 points, do not edit
training_complete = (
    len(EPISODE_RETURNS) == 200
    and len(LOSS_HISTORY) > 0
    and np.all(np.isfinite(EPISODE_RETURNS))
    and np.all(np.isfinite(LOSS_HISTORY))
)
evaluation_complete = (
    len(EVALUATION_RETURNS) == 10
    and np.all(np.isfinite(EVALUATION_RETURNS))
    and all(0.0 <= value <= 500.0 for value in EVALUATION_RETURNS)
)

record_check("G1 training outputs", training_complete, 5,
             "Run the complete training cell and preserve its outputs.")
record_check("G2 evaluation outputs", evaluation_complete, 5,
             "Run 10 greedy evaluation episodes and preserve their outputs.")

if training_complete:
    best_twenty_episode_mean = float(np.max(ma_returns)) if len(ma_returns) else float("nan")
    print(f"Best 20 episode training mean, diagnostic only: {best_twenty_episode_mean:.1f}")


## Part H Objective concept check, 5 points

For each question, replace `"?"` with `"A"`, `"B"`, or `"C"`.

1. Why does DQN use replay memory?  
   A. To make rewards larger  
   B. To reduce temporal correlation and reuse transitions  
   C. To eliminate exploration

2. Why does DQN use a target network?  
   A. To make the bootstrap target change more slowly  
   B. To increase the action dimension  
   C. To remove the discount factor

3. What should a CartPole Q network output for one state?  
   A. One probability  
   B. Four next state values  
   C. One Q value for each of the two actions

4. Which flag masks the bootstrap term in this notebook?  
   A. True termination  
   B. Time limit truncation  
   C. Either flag always masks it

5. What does \(\epsilon=0\) mean during evaluation?  
   A. Always random  
   B. Greedy action selection  
   C. No action is selected

In [ ]:
# TODO H1 through H5, enter A, B, or C
ANSWER_1 = "?"
ANSWER_2 = "?"
ANSWER_3 = "?"
ANSWER_4 = "?"
ANSWER_5 = "?"


In [ ]:
# CHECK H, 5 points, do not edit
concept_answers = [ANSWER_1, ANSWER_2, ANSWER_3, ANSWER_4, ANSWER_5]
correct_answers = ["B", "A", "C", "A", "B"]

for index, (student_answer, correct_answer) in enumerate(
    zip(concept_answers, correct_answers), start=1
):
    record_check(
        f"H{index} concept question",
        str(student_answer).strip().upper() == correct_answer,
        1,
        "Review the DQN data flow and the termination mask.",
    )


## Final grading summary

Run the cell below immediately before submission. The table reports the public checks that the instructor can verify quickly.

In [ ]:
# FINAL CHECK, do not edit
earned_points = sum(
    item["points"] for item in LAB3_CHECKS.values() if item["passed"]
)
possible_points = sum(item["points"] for item in LAB3_CHECKS.values())

print("=" * 64)
print("AE 696 LAB 3 PUBLIC CHECK SUMMARY")
print("=" * 64)
for name, item in LAB3_CHECKS.items():
    status = "PASS" if item["passed"] else "FAIL"
    awarded = item["points"] if item["passed"] else 0
    print(f"{status:4s}  {awarded:2d}/{item['points']:2d}  {name}")
print("=" * 64)
print(f"PUBLIC CHECK SCORE: {earned_points}/{possible_points}")

if earned_points == possible_points == 100:
    print("All public checks passed. The notebook is ready to submit.")
else:
    print("Fix failed checks, restart the kernel, and run all cells again.")


## Optional extension, not graded

After the required work is complete, try one extension:

* Replace the vanilla target with Double DQN action selection.
* Compare hidden dimensions 32, 64, and 128.
* Repeat training with three seeds and report the mean evaluation return.

Do not change the required cells or public checks.